# 04 — Feature Selection

**Primary:** Zhilin Zhang  
**Support:** Tianyi Qin

This notebook uses the same processed rows, train/test split and full feature set as the modelling notebook.

Required outputs:
- one **embedded** method;
- one **filter** method;
- top 3 features from each with scores;
- an explicit comparison of disagreement;
- one concrete hard-case listing ID with actual attributes.

## 1. Imports and data

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42

REPO_ROOT = Path("..").resolve() if Path("../data").exists() else Path(".").resolve()
DATA_PATH = REPO_ROOT / "data" / "processed_listings.csv"
MODEL_SUMMARY_PATH = REPO_ROOT / "output" / "tables" / "model_summary.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError("Generate data/processed_listings.csv with the preprocessing stage first.")

if not MODEL_SUMMARY_PATH.exists():
    raise FileNotFoundError("Generate the modelling result table before this development notebook.")

df = pd.read_csv(DATA_PATH)
model_summary = pd.read_csv(MODEL_SUMMARY_PATH)

print("Processed shape:", df.shape)
print("Split counts:", df["split"].value_counts().to_dict())

Processed shape: (14472, 20)
Split counts: {'train': 11577, 'test': 2895}


## 2. Reuse the exact modelling feature set and training rows

In [2]:
FEATURES = [
    "accommodates",
    "bedrooms",
    "beds",
    "bathrooms",
    "distance_cbd_km",
    "amenity_count",
    "has_pool",
    "has_free_parking",
    "has_air_conditioning",
    "has_dedicated_workspace",
    "has_washer",
    "has_dryer",
]

TARGET = "high_price"
ID_COL = "id"

missing = [c for c in FEATURES + [TARGET, ID_COL, "split"] if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

train_df = (
    df.loc[df["split"].eq("train")].sort_values("id", kind="stable").reset_index(drop=True)
)
test_df = (
    df.loc[df["split"].eq("test")].sort_values("id", kind="stable").reset_index(drop=True)
)

X_train = train_df[FEATURES]
y_train = train_df[TARGET].astype(int)

print("Training rows:", len(train_df))
print("Target counts:", y_train.value_counts().sort_index().to_dict())

Training rows: 11577
Target counts: {0: 8683, 1: 2894}


## 3. Embedded method — tuned Decision Tree feature importance

The best hyperparameters selected for the full-feature Decision Tree in the modelling stage are reused here. This keeps the embedded ranking traceable to the model comparison rather than fitting an unrelated tree.


In [3]:
tree_row = model_summary.loc[
    (model_summary["model"] == "DecisionTree")
    & (model_summary["feature_set"] == "size_location_amenities")
]

if len(tree_row) != 1:
    raise ValueError("Expected exactly one full-feature DecisionTree result from modelling.")

best_params_raw = tree_row.iloc[0]["best_params"]
best_params = json.loads(best_params_raw)

# GridSearchCV stores pipeline parameter names such as model__max_depth.
tree_kwargs = {
    key.replace("model__", ""): value
    for key, value in best_params.items()
    if key.startswith("model__")
}
tree_kwargs["random_state"] = RANDOM_STATE

imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=FEATURES,
    index=X_train.index,
)

tree = DecisionTreeClassifier(**tree_kwargs)
tree.fit(X_train_imp, y_train)

embedded_scores = pd.Series(
    tree.feature_importances_,
    index=FEATURES,
    name="embedded_score",
).sort_values(ascending=False)

embedded_top3 = embedded_scores.head(3)
display(embedded_top3.to_frame())

,embedded_score
bedrooms,0.780613
distance_cbd_km,0.080485
bathrooms,0.059686


## 4. Filter method — Mutual Information

Mutual Information is calculated on the same training rows. Binary amenity indicators are declared discrete; the remaining numeric/count variables are treated as continuous. Median imputation is fitted only on training data.

In [4]:
DISCRETE_FEATURES = {
    "has_pool",
    "has_free_parking",
    "has_air_conditioning",
    "has_dedicated_workspace",
    "has_washer",
    "has_dryer",
}

discrete_mask = np.array([f in DISCRETE_FEATURES for f in FEATURES], dtype=bool)

mi_values = mutual_info_classif(
    X_train_imp,
    y_train,
    discrete_features=discrete_mask,
    random_state=RANDOM_STATE,
)

filter_scores = pd.Series(
    mi_values,
    index=FEATURES,
    name="filter_mi_score",
).sort_values(ascending=False)

filter_top3 = filter_scores.head(3)
display(filter_top3.to_frame())

,filter_mi_score
bedrooms,0.142634
accommodates,0.127700
bathrooms,0.107099


## 5. Compare the two top-3 lists

The comparison table preserves both ranks and scores. The report discussion should explain any disagreement using these actual values and the different mechanics of embedded vs filter selection.

In [5]:
rank_table = pd.DataFrame({
    "feature": FEATURES,
    "embedded_score": embedded_scores.reindex(FEATURES).values,
    "filter_mi_score": filter_scores.reindex(FEATURES).values,
})

rank_table["embedded_rank"] = rank_table["embedded_score"].rank(
    method="min", ascending=False
).astype(int)
rank_table["filter_rank"] = rank_table["filter_mi_score"].rank(
    method="min", ascending=False
).astype(int)
rank_table["rank_difference"] = (
    rank_table["embedded_rank"] - rank_table["filter_rank"]
).abs()

rank_table = rank_table.sort_values(
    ["embedded_rank", "filter_rank", "feature"]
).reset_index(drop=True)

display(rank_table)

top3_comparison = pd.DataFrame({
    "embedded_feature": embedded_top3.index.tolist(),
    "embedded_score": embedded_top3.values.tolist(),
    "filter_feature": filter_top3.index.tolist(),
    "filter_score": filter_top3.values.tolist(),
})
display(top3_comparison)

,feature,embedded_score,filter_mi_score,embedded_rank,filter_rank,rank_difference
0,bedrooms,0.780613,0.142634,1,1,0
1,distance_cbd_km,0.080485,0.022122,2,6,4
2,bathrooms,0.059686,0.107099,3,3,0
3,accommodates,0.059537,0.127700,4,2,2
4,amenity_count,0.016097,0.022659,5,5,0
5,beds,0.001791,0.094425,6,4,2
6,has_free_parking,0.001791,0.012479,7,7,0
7,has_dryer,0.000000,0.004514,8,8,0
8,has_washer,0.000000,0.004036,8,9,1
9,has_dedicated_workspace,0.000000,0.001689,8,10,2


,embedded_feature,embedded_score,filter_feature,filter_score
0,bedrooms,0.780613,bedrooms,0.142634
1,distance_cbd_km,0.080485,accommodates,0.127700
2,bathrooms,0.059686,bathrooms,0.107099


## 6. Hard-case listing

A hard case should be difficult for the fitted model, not merely extreme on one feature. The procedure below evaluates the held-out test set with the tuned full-feature Decision Tree and selects the **highest-confidence misclassification**. The exported row records its true label, predicted label, prediction confidence, high-price probability, price, all model features, and its exact price margin above/below the training-Q75 decision boundary.

This is a post-model diagnostic. It is not used for training, tuning, or choosing the model.


In [6]:
X_test_imp = pd.DataFrame(
    imputer.transform(test_df[FEATURES]),
    columns=FEATURES,
    index=test_df.index,
)
test_prediction = tree.predict(X_test_imp).astype(int)
test_probability = tree.predict_proba(X_test_imp)
predicted_probability = test_probability[
    np.arange(len(test_prediction)), test_prediction
]

hard_candidates = test_df[[ID_COL, TARGET, "price_clean"] + FEATURES].copy()
hard_candidates["predicted_high_price"] = test_prediction
hard_candidates["probability_high_price"] = test_probability[:, 1]
hard_candidates["predicted_class_probability"] = predicted_probability
hard_candidates["prediction_correct"] = (
    hard_candidates[TARGET].to_numpy() == test_prediction
)
hard_candidates = hard_candidates.loc[~hard_candidates["prediction_correct"]]

if hard_candidates.empty:
    raise ValueError("The held-out test set contains no Decision Tree misclassification.")

hard_idx = hard_candidates["predicted_class_probability"].idxmax()
hard_case = hard_candidates.loc[[hard_idx]].copy()
hard_case.insert(1, "split", "test")
hard_case.insert(
    2,
    "hard_case_reason",
    "highest-confidence misclassification by tuned full-feature Decision Tree",
)
training_price_q75 = float(train_df["price_clean"].quantile(0.75))
hard_case.insert(3, "training_price_q75", training_price_q75)
hard_case.insert(
    4,
    "price_margin_from_training_q75",
    hard_case["price_clean"] - training_price_q75,
)
hard_case.insert(
    5,
    "boundary_case_note",
    "True high-price listing close to the training-Q75 threshold.",
)

display(hard_case.T)


,65
id,6520432
split,test
hard_case_reason,highest-confidence misclassification by tuned ...
training_price_q75,385.15
price_margin_from_training_q75,14.1
boundary_case_note,True high-price listing close to the training-...
high_price,1
price_clean,399.25
accommodates,2
bedrooms,1.0


## 7. Concrete opposite-ranking scenario

Zero-importance ties can create a large numerical rank gap without a meaningful substantive disagreement. The selection below therefore requires both methods to assign positive scores and looks for a feature that the filter ranks above the embedded tree. Ties are resolved toward the smaller embedded importance, highlighting a genuine redundancy/conditional-importance contrast.


In [7]:
meaningful_disagreements = rank_table.loc[
    (rank_table["embedded_score"] > 0)
    & (rank_table["filter_mi_score"] > 0)
    & (rank_table["filter_rank"] < rank_table["embedded_rank"])
].copy()
meaningful_disagreements["filter_over_embedded_rank_gap"] = (
    meaningful_disagreements["embedded_rank"]
    - meaningful_disagreements["filter_rank"]
)

if meaningful_disagreements.empty:
    raise ValueError("No positive-score filter-over-embedded disagreement found.")

largest_disagreement = meaningful_disagreements.sort_values(
    ["filter_over_embedded_rank_gap", "embedded_score", "filter_mi_score"],
    ascending=[False, True, False],
).iloc[[0]]

display(largest_disagreement)


,feature,embedded_score,filter_mi_score,embedded_rank,filter_rank,rank_difference,filter_over_embedded_rank_gap
5,beds,0.001791,0.094425,6,4,2,2


## 8. Save reproducible feature-selection outputs

In [8]:
TABLE_OUT = REPO_ROOT / "output" / "tables"
TABLE_OUT.mkdir(parents=True, exist_ok=True)

rank_table.to_csv(TABLE_OUT / "feature_selection_rankings.csv", index=False)
top3_comparison.to_csv(TABLE_OUT / "feature_selection_top3.csv", index=False)
hard_case.to_csv(TABLE_OUT / "feature_selection_hard_case.csv", index=False)
largest_disagreement.to_csv(
    TABLE_OUT / "feature_selection_largest_rank_disagreement.csv",
    index=False,
)

print("Saved:")
for name in [
    "feature_selection_rankings.csv",
    "feature_selection_top3.csv",
    "feature_selection_hard_case.csv",
    "feature_selection_largest_rank_disagreement.csv",
]:
    print(" -", (TABLE_OUT / name).relative_to(REPO_ROOT))

Saved:
 - output/tables/feature_selection_rankings.csv
 - output/tables/feature_selection_top3.csv
 - output/tables/feature_selection_hard_case.csv
 - output/tables/feature_selection_largest_rank_disagreement.csv


## 9. Evidence checklist

Before the group-written report is finalised, verify:
- embedded top 3 + scores are quoted correctly;
- filter top 3 + scores are quoted correctly;
- disagreement is explained using the actual rank table;
- the hard-case listing ID and relevant actual values are quoted;
- the opposite-ranking scenario names a real feature from this group's feature set;
- feature-selection limitations are specific to the observed results, not generic.